# Kampala Air Quality and TB Analysis - Per-Year Pipeline


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns
from datetime import date
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import sys
sys.path.append('./src')

from simplified_data_processor import SimplifiedDataProcessor
from intervention_simulator import InterventionSimulator
from time_series_modeler import TimeSeriesModeler
from visualizer import Visualizer
from report_generator import ReportGenerator

# Set up plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All modules imported successfully")

import warnings
warnings.filterwarnings('ignore')


In [ ]:
resp_df = pd.read_excel('data/Data RTI DHIS2.xls', sheet_name='MOH - Uganda, Kampala', header=2)
clim_df = pd.read_csv('data/epi_weekly_climate data_kampala.csv')
plo_df = pd.read_csv('data/kampala_weekly_PM2.5 climate data averaged_across_sites.csv')

main_df = pd.merge(
    clim_df,
    plo_df,
    left_on=['epi_year', 'epi_week'],
    right_on=['year', 'week'],
    how='inner'
)

drop_cols = ['year', 'week']
main_df.drop(columns=drop_cols, inplace=True)

# Function to get the last day (Sunday) of each ISO week
def get_last_day_of_week(year, week):
    return date.fromisocalendar(int(year), int(week), 7)

# Apply function to create a proper date column
main_df['date'] = main_df.apply(
    lambda row: get_last_day_of_week(row['epi_year'], row['epi_week']),
    axis=1
)

main_df['date'] = pd.to_datetime(main_df['date'])   # ensure datetime dtype

# Extract the last date (Sunday) from periodname
resp_df['date'] = resp_df['periodname'].str.extract(r'-\s*(\d{4}-\d{2}-\d{2})')
resp_df['date'] = pd.to_datetime(resp_df['date'], errors='coerce')

# Keep ALL columns (not just TB)
resp_df = resp_df.drop(columns=['periodname'])  # drop only the text period column


In [ ]:

# --- Merge climate + TB + other health indicators ---
merged_df = pd.merge(main_df, resp_df, on='date', how='inner')


# Rename climate columns to match processor expectations
rename_map = {
    'weekly_pm25_avg': 'pm2_5',
    'weekly_temp_avg': 'avgtemp',
    'weekly_humidity_avg': 'avghumidity',
    'Pulmonary TB cases': 'TB',
    'Avg_Temp(C)': 'avgtemp_sat',
    'AvgWindspeed(m/s)': 'windspeed',
    'AvgPrecipitation(mm/day)': 'precip',
    'AvgRelative_Humidity(%)': 'humidity_sat',
    'ILI Cases': 'ILI',
    'ILD deaths': 'ILD_deaths',
    'Severe pneumonia under 5 Cases': 'pneumonia_u5',
    'Severe pneumonia under 5 deaths': 'pneumonia_u5_deaths',
    'SARI Cases': 'SARI',
    'SARI Deaths': 'SARI_deaths'
}

merged_df.rename(columns=rename_map, inplace=True)

In [ ]:
# Select the columns you want to impute
cols_to_impute = ['pm2_5', 'avgtemp', 'avghumidity', 'TB']

imputer = SimpleImputer(strategy='mean')  # or 'median', 'most_frequent'
merged_df[cols_to_impute] = imputer.fit_transform(merged_df[cols_to_impute])

# Save merged dataset
merged_df.to_csv('data/merged_climate_tb_data.csv', index=False)

In [ ]:
years_to_plot = [2020, 2021, 2022, 2024]

fig, axes = plt.subplots(len(years_to_plot), 1, figsize=(14, 18), sharex=True)

for i, year in enumerate(years_to_plot):
    year_sample_df = main_df[main_df['epi_year'] == year]
    
    axes[i].plot(
        year_sample_df['epi_week'], 
        year_sample_df['Avg_Temp(C)'], 
        marker='o', 
        label=f'Climate Avg Temp (°C) - Satellite Data ({year})'
    )
    
    axes[i].plot(
        year_sample_df['epi_week'], 
        year_sample_df['weekly_temp_avg'], 
        marker='s', 
        label=f'PM2.5 Weekly Temp Avg (°C) - Ground Station Data ({year})'
    )
    
    axes[i].set_title(f'Weekly Temperature Comparison - {year}')
    axes[i].set_ylabel('Temperature (°C)')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

# Shared x-axis label
plt.xlabel('Epi Week')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ## Step 1: Load and Process Data

# %%
# Initialize data processor
processor = SimplifiedDataProcessor(data_dir='../data')

# Load the merged data
data = processor.load_data('./data/merged_climate_tb_data.csv')
print(f"✓ Loaded data: {len(data)} records")

In [ ]:
# Add lag features and rolling averages
processed_data = processor.add_lag_features()

# Add extra engineered features so they pass through the same initial pipeline
processed_data['date'] = pd.to_datetime(processed_data['date'], errors='coerce', dayfirst=True)
processed_data['temp_humidity_interaction'] = processed_data['avgtemp'] * processed_data['avghumidity']
processed_data['temp_sat_gap'] = processed_data['avgtemp'] - processed_data.get('avgtemp_sat', processed_data['avgtemp'])
processed_data['humidity_sat_gap'] = processed_data['avghumidity'] - processed_data.get('humidity_sat', processed_data['avghumidity'])
processed_data['precip_rolling_4'] = processed_data['precip'].rolling(window=4, min_periods=1).mean()
processed_data['windspeed_rolling_4'] = processed_data['windspeed'].rolling(window=4, min_periods=1).mean()
processed_data['pm25_temp_ratio'] = processed_data['pm2_5'] / (processed_data['avgtemp'].abs() + 1)
processed_data['pm25_humidity_ratio'] = processed_data['pm2_5'] / (processed_data['avghumidity'].abs() + 1)
processed_data['tb_pct_change_1'] = processed_data['TB'].pct_change().replace([np.inf, -np.inf], np.nan)
processed_data['tb_diff_1'] = processed_data['TB'].diff()
processed_data['sin_week'] = np.sin(2 * np.pi * processed_data['week_of_year'] / 52)
processed_data['cos_week'] = np.cos(2 * np.pi * processed_data['week_of_year'] / 52)
processed_data['sin_month'] = np.sin(2 * np.pi * processed_data['month'] / 12)
processed_data['cos_month'] = np.cos(2 * np.pi * processed_data['month'] / 12)

print(f"✓ Added features: {processed_data.shape[1]} total columns")

In [ ]:
# Get summary statistics
summary = processor.get_summary_statistics()
print("\nData Summary:")
print(f"- Date range: {summary['date_range']}")
print(f"- Average PM2.5: {summary['pm25_statistics']['mean']:.2f} μg/m³")
print(f"- Total TB cases: {summary['tb_statistics']['total_cases']}")
print(f"- PM2.5-TB correlation: {summary['correlation_pm25_tb']:.3f}")


In [ ]:
# Display first few rows including diseases and engineered features
disease_cols = [
    col for col in ['TB', 'ILI', 'ILD_deaths', 'pneumonia_u5', 'pneumonia_u5_deaths', 'SARI', 'SARI_deaths']
    if col in processed_data.columns
]
preview_cols = [
    'date', 'pm2_5', 'avgtemp', 'avghumidity', 'windspeed', 'precip'
] + disease_cols + [
    'temp_humidity_interaction', 'temp_sat_gap', 'humidity_sat_gap',
    'precip_rolling_4', 'windspeed_rolling_4', 'pm25_temp_ratio',
    'pm25_humidity_ratio', 'tb_pct_change_1', 'tb_diff_1',
    'sin_week', 'cos_week', 'sin_month', 'cos_month'
]
preview_cols = [col for col in preview_cols if col in processed_data.columns]
processed_data[preview_cols].head(10)


In [ ]:
# Time series overview for air quality, diseases, and weather
disease_cols = [
    col for col in ['TB', 'ILI', 'ILD_deaths', 'pneumonia_u5', 'pneumonia_u5_deaths', 'SARI', 'SARI_deaths']
    if col in processed_data.columns
]

fig, axes = plt.subplots(4, 1, figsize=(15, 16), sharex=True)

axes[0].plot(processed_data['date'], processed_data['pm2_5'], color='blue', alpha=0.8)
axes[0].axhline(y=15, color='red', linestyle='--', label='WHO Guideline (24-hour average)')
axes[0].axhline(y=5, color='orange', linestyle='--', label='WHO Guideline (annual average)')
axes[0].set_title('PM2.5 Levels in Kampala')
axes[0].set_ylabel('PM2.5 (?g/m?)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for disease in disease_cols:
    axes[1].plot(processed_data['date'], processed_data[disease], alpha=0.75, linewidth=1.6, label=disease)
axes[1].set_title('Disease Burden Over Time')
axes[1].set_ylabel('Cases / Deaths')
axes[1].legend(loc='upper left', ncol=2)
axes[1].grid(True, alpha=0.3)

ax2_twin = axes[2].twinx()
axes[2].plot(processed_data['date'], processed_data['avgtemp'], color='orange', label='Temperature')
ax2_twin.plot(processed_data['date'], processed_data['avghumidity'], color='cyan', label='Humidity')
axes[2].set_title('Weather Conditions')
axes[2].set_ylabel('Temperature (?C)', color='orange')
ax2_twin.set_ylabel('Humidity (%)', color='cyan')
axes[2].grid(True, alpha=0.3)

engineered_plot_cols = [
    col for col in ['temp_humidity_interaction', 'precip_rolling_4', 'windspeed_rolling_4', 'pm25_temp_ratio']
    if col in processed_data.columns
]
engineered_scaled = processed_data[engineered_plot_cols].copy()
for col in engineered_plot_cols:
    col_std = engineered_scaled[col].std()
    if pd.notna(col_std) and col_std != 0:
        engineered_scaled[col] = (engineered_scaled[col] - engineered_scaled[col].mean()) / col_std
    else:
        engineered_scaled[col] = 0
for col in engineered_plot_cols:
    axes[3].plot(processed_data['date'], engineered_scaled[col], linewidth=1.5, label=col)
axes[3].set_title('Scaled Engineered Feature Trends')
axes[3].set_ylabel('Z-score')
axes[3].set_xlabel('Date')
axes[3].legend(loc='upper left', ncol=2)
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Correlation analysis across diseases and engineered features
disease_cols = [
    col for col in ['TB', 'ILI', 'ILD_deaths', 'pneumonia_u5', 'pneumonia_u5_deaths', 'SARI', 'SARI_deaths']
    if col in processed_data.columns
]
feature_analysis_cols = [
    col for col in [
        'pm2_5', 'avgtemp', 'avghumidity', 'avgtemp_sat', 'humidity_sat',
        'windspeed', 'precip', 'pm25_lag_1', 'pm25_lag_2', 'tb_lag_1', 'tb_lag_2',
        'tb_lag_3', 'tb_lag_4', 'pm25_ma_7', 'tb_ma_7', 'pm25_ma_14', 'tb_ma_14',
        'pm25_ma_30', 'tb_ma_30', 'temp_humidity_interaction', 'temp_sat_gap',
        'humidity_sat_gap', 'precip_rolling_4', 'windspeed_rolling_4',
        'pm25_temp_ratio', 'pm25_humidity_ratio', 'tb_pct_change_1', 'tb_diff_1',
        'sin_week', 'cos_week', 'sin_month', 'cos_month', 'time', 'intervention',
        'time_since_intervention'
    ]
    if col in processed_data.columns
]

feature_corr_matrix = processed_data[feature_analysis_cols].corr()
plt.figure(figsize=(16, 12))
sns.heatmap(feature_corr_matrix, cmap='coolwarm', center=0)
plt.title('Correlation Matrix for Environmental and Engineered Features')
plt.tight_layout()
plt.show()

disease_feature_corr = processed_data[disease_cols + feature_analysis_cols].corr().loc[disease_cols, feature_analysis_cols]
plt.figure(figsize=(18, max(4, len(disease_cols) * 1.2)))
sns.heatmap(disease_feature_corr, annot=True, cmap='vlag', center=0, fmt='.2f')
plt.title('Disease-to-Feature Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Disease-level feature ranking and pairwise relationships
disease_cols = [
    col for col in ['TB', 'ILI', 'ILD_deaths', 'pneumonia_u5', 'pneumonia_u5_deaths', 'SARI', 'SARI_deaths']
    if col in processed_data.columns
]
feature_analysis_cols = [
    col for col in [
        'pm2_5', 'avgtemp', 'avghumidity', 'avgtemp_sat', 'humidity_sat',
        'windspeed', 'precip', 'pm25_lag_1', 'pm25_lag_2', 'tb_lag_1', 'tb_lag_2',
        'tb_lag_3', 'tb_lag_4', 'pm25_ma_7', 'tb_ma_7', 'pm25_ma_14', 'tb_ma_14',
        'pm25_ma_30', 'tb_ma_30', 'temp_humidity_interaction', 'temp_sat_gap',
        'humidity_sat_gap', 'precip_rolling_4', 'windspeed_rolling_4',
        'pm25_temp_ratio', 'pm25_humidity_ratio', 'tb_pct_change_1', 'tb_diff_1',
        'sin_week', 'cos_week', 'sin_month', 'cos_month', 'time', 'intervention',
        'time_since_intervention'
    ]
    if col in processed_data.columns
]

disease_feature_corr = processed_data[disease_cols + feature_analysis_cols].corr().loc[disease_cols, feature_analysis_cols]
strongest_links = (
    disease_feature_corr.abs()
    .stack()
    .reset_index(name='abs_correlation')
    .rename(columns={'level_0': 'disease', 'level_1': 'feature'})
    .sort_values(['disease', 'abs_correlation'], ascending=[True, False])
    .groupby('disease')
    .head(5)
)
strongest_links['correlation'] = strongest_links.apply(
    lambda row: disease_feature_corr.loc[row['disease'], row['feature']],
    axis=1
)
print('Top 5 absolute correlations per disease')
print(strongest_links[['disease', 'feature', 'correlation', 'abs_correlation']].to_string(index=False))

summary_stats = processed_data[disease_cols + feature_analysis_cols].describe().T[['mean', 'std', 'min', 'max']].round(2)
summary_stats.head(20)


In [ ]:
# --- 1. Data Preprocessing ---

# Convert 'date' to datetime and set as index
data_itsa = processed_data.copy()
data_itsa['date'] = pd.to_datetime(data['date'], format='%d/%m/%Y')
data_itsa = data_itsa.sort_values('date').set_index('date')

In [ ]:
# --- 2. Handle Missing Data ---

# Check for missing values before handling them
print("--- Missing Values Before Handling ---")
print(data_itsa.isnull().sum())

# Use forward fill to handle missing values, a common method for time series
data_itsa.fillna(value=0, inplace=True)

# Verify that missing values are handled
print("\n--- Missing Values After Forward Fill ---")
print(data.isnull().sum())


In [ ]:
# --- 3. Define the Intervention Point ---

# Find the halfway point in the data
intervention_point = len(data_itsa) // 2
intervention_date = data_itsa.index[intervention_point]

print(f"\nThe dataset has {len(data_itsa)} data points.")
print(f"The intervention is set at index {intervention_point}, which correspondate to the date: {intervention_date.date()}")


In [ ]:
# --- 4. Create Interrupted Time Series (ITS) Variables ---

# Time variable: a sequence from 1 to the number of observations
data_itsa['time'] = np.arange(1, len(data_itsa) + 1)

# Intervention variable: 0 before the intervention, 1 after
data_itsa['intervention'] = (data_itsa.index >= intervention_date).astype(int)

# Time since intervention: 0 before, and counts up from 1 after the intervention
data_itsa['time_since_intervention'] = 0
post_intervention_indices = data_itsa[data_itsa['intervention'] == 1].index
time_since_intervention_values = np.arange(len(post_intervention_indices))
data_itsa.loc[post_intervention_indices, 'time_since_intervention'] = time_since_intervention_values

In [ ]:
# --- 5. LSTM Modeling for predicted_tb ---

feature_cols = [
    'TB', 'pm2_5', 'avgtemp', 'avghumidity', 'avgtemp_sat', 'humidity_sat',
    'windspeed', 'precip', 'pm25_lag_1', 'pm25_lag_2', 'tb_lag_1', 'tb_lag_2',
    'tb_lag_3', 'tb_lag_4', 'pm25_ma_7', 'tb_ma_7', 'pm25_ma_14', 'tb_ma_14',
    'pm25_ma_30', 'tb_ma_30', 'temp_humidity_interaction', 'temp_sat_gap',
    'humidity_sat_gap', 'precip_rolling_4', 'windspeed_rolling_4',
    'pm25_temp_ratio', 'pm25_humidity_ratio', 'tb_pct_change_1', 'tb_diff_1',
    'sin_week', 'cos_week', 'sin_month', 'cos_month', 'time', 'intervention',
    'time_since_intervention'
]

sequence_length = 8
lstm_df = data_itsa[feature_cols].copy().fillna(0)

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
scaled_features = feature_scaler.fit_transform(lstm_df)
scaled_target = target_scaler.fit_transform(data_itsa[['TB']])

def build_sequences(features, target, seq_len):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(features)):
        X_seq.append(features[i-seq_len:i])
        y_seq.append(target[i])
    return np.array(X_seq), np.array(y_seq)

X_all, y_all = build_sequences(scaled_features, scaled_target, sequence_length)
train_end = max(sequence_length + 1, intervention_point)
X_train, y_train = X_all[:train_end-sequence_length], y_all[:train_end-sequence_length]

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(sequence_length, X_all.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
history = lstm_model.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
    callbacks=[early_stopping]
)

pred_scaled = lstm_model.predict(X_all, verbose=0)
pred_tb = target_scaler.inverse_transform(pred_scaled).flatten()

data_itsa['predicted_tb'] = data_itsa['TB'].astype(float)
data_itsa.iloc[sequence_length:, data_itsa.columns.get_loc('predicted_tb')] = pred_tb

counterfactual_features = lstm_df.copy()
counterfactual_features.loc[counterfactual_features.index >= intervention_date, 'intervention'] = 0
counterfactual_features.loc[counterfactual_features.index >= intervention_date, 'time_since_intervention'] = 0
scaled_counterfactual = feature_scaler.transform(counterfactual_features.fillna(0))
X_counterfactual, _ = build_sequences(scaled_counterfactual, scaled_target, sequence_length)
counter_scaled = lstm_model.predict(X_counterfactual, verbose=0)
counter_tb = target_scaler.inverse_transform(counter_scaled).flatten()

data_itsa['counterfactual_tb'] = data_itsa['TB'].astype(float)
data_itsa.iloc[sequence_length:, data_itsa.columns.get_loc('counterfactual_tb')] = counter_tb

actual_for_metrics = data_itsa['TB'].iloc[sequence_length:]
pred_for_metrics = data_itsa['predicted_tb'].iloc[sequence_length:]
rmse = np.sqrt(mean_squared_error(actual_for_metrics, pred_for_metrics))
mae = mean_absolute_error(actual_for_metrics, pred_for_metrics)

print("\n--- LSTM Model Summary ---")
print(f"Features used: {len(feature_cols)}")
print(f"Sequence length: {sequence_length} weeks")
print(f"Training windows: {len(X_train)}")
print(f"Best validation loss: {min(history.history['val_loss']):.6f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

In [ ]:
# Make sure the index is datetime
data_itsa.index = pd.to_datetime(data_itsa.index, errors='coerce')

# Add a year column from the index for per-year analysis
data_itsa['year'] = data_itsa.index.year
years = sorted(data_itsa['year'].dropna().unique())


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

# Per-year ITS plots
for year in years:
    year_df = data_itsa[data_itsa['year'] == year]

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.scatter(year_df.index, year_df['TB'], color='black', alpha=0.6, label='Observed TB Cases', s=15)

    pre_data = year_df[year_df['intervention'] == 0]
    post_data = year_df[year_df['intervention'] == 1]
    ax.plot(pre_data.index, pre_data['predicted_tb'], color='blue', linewidth=2, label='Pre-intervention Trend')
    ax.plot(post_data.index, post_data['predicted_tb'], color='red', linewidth=2, label='Post-intervention Trend')
    ax.plot(post_data.index, post_data['counterfactual_tb'], color='green', linestyle='--', linewidth=2, label='Counterfactual (No Intervention)')

    if intervention_date.year == year:
        ax.axvline(x=intervention_date, color='purple', linestyle=':', linewidth=2.5, label=f'Intervention: {intervention_date.date()}')

    ax.set_title(f'ITS Analysis of TB Cases - {year}', fontsize=18, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Number of TB Cases', fontsize=12)
    ax.legend(loc='upper left', fontsize=10)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# Monthly patterns across all diseases and key features
processed_data['month'] = processed_data['date'].dt.month
disease_cols = [
    col for col in ['TB', 'ILI', 'ILD_deaths', 'pneumonia_u5', 'pneumonia_u5_deaths', 'SARI', 'SARI_deaths']
    if col in processed_data.columns
]
monthly_feature_cols = [
    col for col in ['pm2_5', 'avgtemp', 'avghumidity', 'windspeed', 'precip']
    if col in processed_data.columns
]
monthly_agg = {col: ['mean', 'std'] for col in monthly_feature_cols}
monthly_agg.update({col: ['mean', 'sum'] for col in disease_cols})
monthly_stats = processed_data.groupby('month').agg(monthly_agg).round(2)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].bar(range(1, 13), monthly_stats['pm2_5']['mean'].values, color='blue', alpha=0.7)
axes[0, 0].errorbar(
    range(1, 13),
    monthly_stats['pm2_5']['mean'].values,
    yerr=monthly_stats['pm2_5']['std'].fillna(0).values,
    fmt='none',
    color='black',
    alpha=0.5
)
axes[0, 0].set_title('Average PM2.5 by Month')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('PM2.5 (?g/m?)')
axes[0, 0].set_xticks(range(1, 13))

for disease in disease_cols:
    axes[0, 1].plot(range(1, 13), monthly_stats[disease]['sum'].values, marker='o', linewidth=2, label=disease)
axes[0, 1].set_title('Monthly Totals by Disease')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Cases / Deaths')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].legend(loc='upper left', ncol=2)
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(range(1, 13), monthly_stats['avgtemp']['mean'].values, 'o-', color='orange', label='Temperature')
axes[1, 0].plot(range(1, 13), monthly_stats['avghumidity']['mean'].values, 'o-', color='cyan', label='Humidity')
axes[1, 0].set_title('Weather Conditions by Month')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Monthly Mean')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

seasonal_feature_cols = [
    col for col in ['windspeed', 'precip']
    if col in processed_data.columns
]
for col in seasonal_feature_cols:
    axes[1, 1].plot(range(1, 13), monthly_stats[col]['mean'].values, marker='o', linewidth=2, label=col)
axes[1, 1].set_title('Additional Seasonal Environmental Features')
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Monthly Mean')
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Seasonal Patterns Across Diseases and Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

monthly_disease_totals = monthly_stats.loc[:, pd.IndexSlice[disease_cols, 'sum']]
monthly_disease_totals.columns = [col for col, _ in monthly_disease_totals.columns]
monthly_disease_totals


In [ ]:
data_project = processed_data.copy()
data_project = data_project[["date", "TB"]]
data_project = data_project.rename(columns={"TB": "y"})
data_project['date'] = pd.to_datetime(data_project['date'], format='%d/%m/%Y')

## Step 5: Time Series Modeling


In [ ]:

# Prepare data for additional time-series model comparison

data_predict = processed_data.copy()
data_predict['date'] = pd.to_datetime(data_predict['date'], errors='coerce')
data_predict = data_predict.sort_values('date')

model_feature_cols = [
    'pm2_5', 'avgtemp', 'avghumidity', 'windspeed', 'precip',
    'temp_humidity_interaction', 'temp_sat_gap', 'humidity_sat_gap',
    'precip_rolling_4', 'windspeed_rolling_4', 'pm25_temp_ratio',
    'pm25_humidity_ratio', 'tb_pct_change_1', 'tb_diff_1',
    'pm25_lag_1', 'pm25_lag_2', 'tb_lag_1', 'tb_lag_2', 'tb_lag_3', 'tb_lag_4',
    'pm25_ma_7', 'tb_ma_7', 'pm25_ma_14', 'tb_ma_14', 'pm25_ma_30', 'tb_ma_30',
    'sin_week', 'cos_week', 'sin_month', 'cos_month'
]

available_model_features = [col for col in model_feature_cols if col in data_predict.columns]
print(f'Available model features: {len(available_model_features)}')
print(available_model_features)


In [ ]:

# Build a compact benchmark using the local time series modeler

ts_modeler = TimeSeriesModeler(data_predict, target_column='TB')
X_ts, y_ts = ts_modeler.prepare_time_series_data(
    window_size=8,
    features=available_model_features,
    target_type='binary'
)

print('Prepared time-series benchmark arrays:')
print('X shape:', X_ts.shape)
print('y shape:', y_ts.shape)


In [ ]:

benchmark_results = ts_modeler.train_models(
    X_ts,
    y_ts,
    models_to_train=['random_forest', 'gradient_boosting', 'svm'],
    cv_splits=3
)

if benchmark_results:
    models_performance = pd.DataFrame([
        {
            'Model': model_name,
            'Accuracy': metrics.get('mean_accuracy', np.nan),
            'F1 Score': metrics.get('mean_f1', np.nan),
            'Precision': metrics.get('mean_precision', np.nan),
            'Recall': metrics.get('mean_recall', np.nan)
        }
        for model_name, metrics in benchmark_results.items()
    ]).sort_values('F1 Score', ascending=False)
else:
    models_performance = pd.DataFrame(columns=['Model', 'Accuracy', 'F1 Score', 'Precision', 'Recall'])

models_performance


In [ ]:

# Compare the benchmark models against the LSTM regression metrics already computed
print('--- Model Comparison Summary ---')
if not models_performance.empty:
    print(models_performance.round(3).to_string(index=False))
else:
    print('No benchmark models were trained.')

print(f'
LSTM Regression RMSE: {rmse:.2f}')
print(f'LSTM Regression MAE: {mae:.2f}')


## Step 4: Intervention Simulation


In [ ]:

# Get baseline metrics
baseline = processor.get_intervention_baseline()
print('Baseline Metrics:')
print(f"- Average PM2.5: {baseline['baseline_pm25']:.2f}")
print(f"- Average TB cases: {baseline['baseline_tb']:.2f}")
print(f"- Total TB cases: {baseline['total_tb_cases']:.2f}")
print(f"- PM2.5-TB correlation: {baseline['correlation']:.3f}")


In [ ]:

# Simulate intervention scenarios
scenarios = {
    'Traffic Control': {'reduction': 0.25, 'cost_factor': 2.5},
    'Green Spaces': {'reduction': 0.15, 'cost_factor': 3.0},
    'Dust Control': {'reduction': 0.20, 'cost_factor': 4.0},
    'Combined': {'reduction': 0.35, 'cost_factor': 5.0}
}

cr_tb = 1.08  # Relative risk per 10 ?g/m?
results = []
for scenario_name, params in scenarios.items():
    reduced_pm25 = baseline['baseline_pm25'] * (1 - params['reduction'])
    pm25_change = reduced_pm25 - baseline['baseline_pm25']
    relative_risk = cr_tb ** (pm25_change / 10)
    tb_cases_baseline = baseline['total_tb_cases']
    tb_cases_with_intervention = tb_cases_baseline * relative_risk
    tb_cases_prevented = tb_cases_baseline - tb_cases_with_intervention
    cost = params['cost_factor'] * 1_000_000
    cost_per_case = cost / tb_cases_prevented if tb_cases_prevented > 0 else np.inf
    results.append({
        'Scenario': scenario_name,
        'PM2.5 Reduction (%)': params['reduction'] * 100,
        'Final PM2.5 (?g/m?)': reduced_pm25,
        'TB Cases Prevented': tb_cases_prevented,
        'Cost Factor': params['cost_factor'],
        'Cost per Case Prevented ($)': cost_per_case
    })

results_df = pd.DataFrame(results)
results_df


In [ ]:

# Visualize intervention impacts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(
    results_df['PM2.5 Reduction (%)'],
    results_df['TB Cases Prevented'],
    s=results_df['Cost Factor'] * 60,
    alpha=0.7,
    c=range(len(results_df)),
    cmap='viridis'
)
for i, txt in enumerate(results_df['Scenario']):
    axes[0].annotate(
        txt,
        (results_df['PM2.5 Reduction (%)'].iloc[i], results_df['TB Cases Prevented'].iloc[i])
    )
axes[0].set_xlabel('PM2.5 Reduction (%)')
axes[0].set_ylabel('TB Cases Prevented')
axes[0].set_title('Intervention Effectiveness')
axes[0].grid(True, alpha=0.3)

axes[1].bar(
    results_df['Scenario'],
    results_df['Cost per Case Prevented ($)'],
    color=['blue', 'green', 'orange', 'red'],
    alpha=0.7
)
axes[1].set_xlabel('Scenario')
axes[1].set_ylabel('Cost per Case Prevented ($)')
axes[1].set_title('Cost-Effectiveness Analysis')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## Step 6: Generate Report


In [ ]:
# Summary Report
print('=' * 60)
print('KAMPALA AIR QUALITY AND RESPIRATORY HEALTH ANALYSIS - SUMMARY REPORT')
print('=' * 60)
print()

disease_cols = [
    col for col in ['TB', 'ILI', 'ILD_deaths', 'pneumonia_u5', 'pneumonia_u5_deaths', 'SARI', 'SARI_deaths']
    if col in processed_data.columns
]
feature_analysis_cols = [
    col for col in [
        'pm2_5', 'avgtemp', 'avghumidity', 'avgtemp_sat', 'humidity_sat',
        'windspeed', 'precip', 'pm25_lag_1', 'pm25_lag_2', 'tb_lag_1', 'tb_lag_2',
        'tb_lag_3', 'tb_lag_4', 'pm25_ma_7', 'tb_ma_7', 'pm25_ma_14', 'tb_ma_14',
        'pm25_ma_30', 'tb_ma_30', 'temp_humidity_interaction', 'temp_sat_gap',
        'humidity_sat_gap', 'precip_rolling_4', 'windspeed_rolling_4',
        'pm25_temp_ratio', 'pm25_humidity_ratio', 'tb_pct_change_1', 'tb_diff_1',
        'sin_week', 'cos_week', 'sin_month', 'cos_month', 'time', 'intervention',
        'time_since_intervention'
    ]
    if col in processed_data.columns
]

disease_totals = processed_data[disease_cols].sum().sort_values(ascending=False)
disease_feature_corr = processed_data[disease_cols + feature_analysis_cols].corr().loc[disease_cols, feature_analysis_cols]
strongest_links = []
for disease in disease_cols:
    strongest_feature = disease_feature_corr.loc[disease].abs().sort_values(ascending=False).index[0]
    strongest_value = disease_feature_corr.loc[disease, strongest_feature]
    strongest_links.append((disease, strongest_feature, strongest_value))

print('1. DATA OVERVIEW')
print('-' * 40)
print(f"   - Analysis Period: {summary['date_range']}")
print(f"   - Total Records: {summary['total_records']}")
print(f"   - Average PM2.5: {summary['pm25_statistics']['mean']:.2f} ?g/m?")
print(f"   - Diseases analyzed: {', '.join(disease_cols)}")
print()

print('2. DISEASE BURDEN')
print('-' * 40)
for disease, total in disease_totals.items():
    print(f"   - Total {disease}: {total:.2f}")
print()

print('3. KEY FEATURE RELATIONSHIPS')
print('-' * 40)
for disease, feature, value in strongest_links:
    print(f"   - {disease}: strongest correlation with {feature} ({value:.3f})")
print(f"   - TB LSTM RMSE: {rmse:.2f}")
print(f"   - TB LSTM MAE: {mae:.2f}")
print()

print('4. INTERVENTION RECOMMENDATIONS')
print('-' * 40)
best_scenario = results_df.loc[results_df['TB Cases Prevented'].idxmax()]
print(f"   - Most Effective: {best_scenario['Scenario']}")
print(f"   - PM2.5 Reduction: {best_scenario['PM2.5 Reduction (%)']:.1f}%")
print(f"   - TB Cases Prevented: {best_scenario['TB Cases Prevented']:.0f}")
print()

print('5. BENCHMARK MODEL PERFORMANCE')
print('-' * 40)
if not models_performance.empty:
    best_model = models_performance.iloc[0]
    print(f"   - Best Benchmark Model: {best_model['Model']}")
    print(f"   - Accuracy: {best_model['Accuracy']:.3f}")
    print(f"   - F1 Score: {best_model['F1 Score']:.3f}")
else:
    print('   - No benchmark classification models were trained.')
print()
print('=' * 60)

results_dir = Path('./results')
results_dir.mkdir(exist_ok=True)
results_df.to_csv(results_dir / 'intervention_scenarios.csv', index=False)
models_performance.to_csv(results_dir / 'model_performance.csv', index=False)
print('? Results saved to ./results/')
